# Griffin Library - Intelligent Book Recommendation System

## Notebook 03 · Exploratory Data Analysis (EDA)

**Goal:** Explore the clean merged dataset to uncover patterns, distributions,  
and insights that will guide feature engineering and model design decisions.

| | Details |
|---|---|
| **Input** | `data/processed/books_merged.csv` |
| **Operations** | Rating distribution · Genre analysis · Text length analysis · Correlation analysis · Tier comparison |
| **Output** | Documented insights + feature engineering decisions |
| **Next Step** | `04_feature_engineering.ipynb` - Build and select features for the recommendation model |

---

In [1]:
import sys
# Dependencies installed via requirements.txt

import pandas as pd
import os

os.chdir(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))

df = pd.read_csv("data/processed/books_merged.csv")

print(f"Dataset shape : {df.shape}")
print(f"Columns       : {list(df.columns)}")
print(f"\nDtypes:")
print(df.dtypes)
print(f"\nNulls:")
print(df.isnull().sum())

Dataset shape : (8577, 12)
Columns       : ['book_id', 'title', 'authors', 'genres', 'avg_rating', 'num_ratings', 'description', 'summary', 'embedding_text', 'tier', 'url', 'num_pages']

Dtypes:
book_id             int64
title                 str
authors               str
genres                str
avg_rating        float64
num_ratings         int64
description           str
summary               str
embedding_text        str
tier                int64
url                   str
num_pages           int64
dtype: object

Nulls:
book_id              0
title                0
authors              0
genres             134
avg_rating           0
num_ratings          0
description          0
summary           6993
embedding_text       0
tier                 0
url                  0
num_pages            0
dtype: int64


---

In [2]:
# ═══════════════════════════════════════════════════════════════
# RATINGS ANALYSIS
# ═══════════════════════════════════════════════════════════════

print("=== Average Rating Distribution ===")
print(df["avg_rating"].describe())

print("\n=== Ratings Count Distribution ===")
print(df["num_ratings"].describe())

print("\n=== Rating Buckets ===")
bins = [0, 3.5, 3.75, 4.0, 4.25, 4.5, 5.0]
labels = ["<3.5", "3.5-3.75", "3.75-4.0", "4.0-4.25", "4.25-4.5", ">4.5"]
df["rating_bucket"] = pd.cut(df["avg_rating"], bins=bins, labels=labels)
print(df["rating_bucket"].value_counts().sort_index())

print("\n=== Popularity Buckets (num_ratings) ===")
pop_bins = [0, 1000, 10000, 100000, 1000000, float("inf")]
pop_labels = ["<1K", "1K-10K", "10K-100K", "100K-1M", ">1M"]
df["popularity_bucket"] = pd.cut(df["num_ratings"], bins=pop_bins, labels=pop_labels)
print(df["popularity_bucket"].value_counts().sort_index())

=== Average Rating Distribution ===
count    8577.000000
mean        4.037948
std         0.266280
min         1.640000
25%         3.870000
50%         4.050000
75%         4.220000
max         5.000000
Name: avg_rating, dtype: float64

=== Ratings Count Distribution ===
count    8.577000e+03
mean     1.006977e+05
std      3.290911e+05
min      5.000000e+01
25%      4.041000e+03
50%      2.309000e+04
75%      7.590800e+04
max      9.278135e+06
Name: num_ratings, dtype: float64

=== Rating Buckets ===
rating_bucket
<3.5         247
3.5-3.75     915
3.75-4.0    2536
4.0-4.25    3099
4.25-4.5    1525
>4.5         255
Name: count, dtype: int64

=== Popularity Buckets (num_ratings) ===
popularity_bucket
<1K         1458
1K-10K      1544
10K-100K    3867
100K-1M     1583
>1M          125
Name: count, dtype: int64


### Ratings Analysis - Key Insights

- **Healthy distribution**: mean rating 4.04, std 0.27 - tight and reliable signal
- **Majority in sweet spot**: 54% of books rated between 3.75 and 4.25 - well-calibrated catalog
- **Extreme ratings are rare**: only 247 books below 3.5 and 255 above 4.5 - no artificial inflation
- **Highly skewed popularity**: median 23K ratings but max 9.2M - a small number of books dominate
- **45% of books have < 10K ratings**: Bayesian smoothing will be critical in feature engineering to avoid overranking obscure books
- **125 blockbuster books (>1M ratings)**: these are cultural landmarks - important anchors for the recommendation system

---

In [3]:
# ═══════════════════════════════════════════════════════════════
# GENRE ANALYSIS
# ═══════════════════════════════════════════════════════════════

from collections import Counter

# Flatten all genres
all_genres = []
for g in df["genres"].dropna():
    all_genres.extend([x.strip() for x in g.split(",")])

genre_counts = Counter(all_genres)

print(f"Total unique genres : {len(genre_counts)}")
print(f"\nTop 20 genres:")
for genre, count in genre_counts.most_common(20):
    bar = "█" * (count // 50)
    print(f"  {genre:<30} {count:>5,}  {bar}")

print(f"\nBooks with no genres: {df['genres'].isna().sum()}")
print(f"Avg genres per book : {df['genres'].dropna().str.split(',').apply(len).mean():.1f}")

Total unique genres : 611

Top 20 genres:
  Fiction                        5,529  ██████████████████████████████████████████████████████████████████████████████████████████████████████████████
  Nonfiction                     2,222  ████████████████████████████████████████████
  Classics                       2,110  ██████████████████████████████████████████
  Fantasy                        2,089  █████████████████████████████████████████
  Romance                        1,481  █████████████████████████████
  Young Adult                    1,474  █████████████████████████████
  Historical Fiction             1,440  ████████████████████████████
  Mystery                        1,325  ██████████████████████████
  Contemporary                   1,246  ████████████████████████
  Audiobook                      1,196  ███████████████████████
  Novels                         1,152  ███████████████████████
  Literature                     1,096  █████████████████████
  Thriller                

### Genre Analysis - Key Insights

- **611 unique genres**: rich and diverse catalog covering virtually every reading taste
- **Fiction dominates**: 5,529 books (64%) - expected for a Goodreads-based dataset
- **Strong non-fiction presence**: 2,222 books - good balance for diverse recommendations
- **6.7 genres per book on average**: rich multi-label signal - ideal for semantic search
- **"Audiobook" appears as genre** (1,196 books): noise tag - should be removed in feature engineering
- **"Novels" and "Historical" are redundant**: overlap with Fiction and Historical Fiction - will be consolidated
- **Only 134 books with no genres** (1.6%): negligible - embedding text handles these via description
- **Top 4 genres** (Fiction, Nonfiction, Classics, Fantasy) cover the core of the catalog

---

In [4]:
# ═══════════════════════════════════════════════════════════════
# TEXT LENGTH ANALYSIS
# ═══════════════════════════════════════════════════════════════

df["desc_length"]   = df["description"].str.len()
df["summary_length"] = df["summary"].str.len()
df["embed_length"]  = df["embedding_text"].str.len()

print("=== Description Length ===")
print(df["desc_length"].describe())

print("\n=== CMU Summary Length (Tier 1 only) ===")
print(df[df["tier"]==1]["summary_length"].describe())

print("\n=== Embedding Text Length ===")
print(df["embed_length"].describe())

print("\n=== Embedding Length Buckets ===")
bins   = [0, 500, 1000, 1500, 2000, float("inf")]
labels = ["<500", "500-1K", "1K-1.5K", "1.5K-2K", ">2K"]
print(pd.cut(df["embed_length"], bins=bins, labels=labels).value_counts().sort_index())

=== Description Length ===
count    8577.000000
mean      948.239944
std       553.127627
min         1.000000
25%       591.000000
50%       854.000000
75%      1176.000000
max      8947.000000
Name: desc_length, dtype: float64

=== CMU Summary Length (Tier 1 only) ===
count     1584.000000
mean      3551.868056
std       3409.480313
min         41.000000
25%       1291.000000
50%       2659.500000
75%       4628.250000
max      35346.000000
Name: summary_length, dtype: float64

=== Embedding Text Length ===
count    8577.000000
mean     1250.647429
std       632.285463
min        43.000000
25%       812.000000
50%      1133.000000
75%      1609.000000
max      9047.000000
Name: embed_length, dtype: float64

=== Embedding Length Buckets ===
embed_length
<500        664
500-1K     2783
1K-1.5K    2581
1.5K-2K    1611
>2K         938
Name: count, dtype: int64


### Text Length Analysis - Key Insights

- **Descriptions average 948 chars**: substantial enough for meaningful semantic embedding
- **CMU summaries average 3,551 chars**: 3.7x longer than descriptions - significant enrichment for Tier 1
- **Embedding text averages 1,251 chars**: well within optimal range for sentence-transformers
- **664 books with embedding < 500 chars**: very short texts - may produce weaker embeddings, worth monitoring
- **CMU summary std is high (3,409)**: wide variance - some summaries are Wikipedia-length (35K chars), truncation to 1,000 chars in Step 5 was the right decision
- **75% of books have embedding > 812 chars**: strong coverage across the catalog

---

In [5]:
# ═══════════════════════════════════════════════════════════════
# CORRELATION ANALYSIS
# ═══════════════════════════════════════════════════════════════

import numpy as np

corr_df = df[["avg_rating", "num_ratings", "desc_length", "embed_length", "tier"]].copy()
corr = corr_df.corr()

print("=== Correlation Matrix ===")
print(corr.round(3))

print("\n=== Key Correlations with avg_rating ===")
rating_corr = corr["avg_rating"].drop("avg_rating").sort_values(ascending=False)
for col, val in rating_corr.items():
    direction = "↑" if val > 0 else "↓"
    print(f"  {col:<20} {val:>7.3f}  {direction}")

print("\n=== Tier Comparison ===")
print(df.groupby("tier")[["avg_rating", "num_ratings", "desc_length"]].mean().round(2))

=== Correlation Matrix ===
              avg_rating  num_ratings  desc_length  embed_length   tier
avg_rating         1.000        0.058        0.006        -0.087  0.166
num_ratings        0.058        1.000       -0.026         0.033 -0.086
desc_length        0.006       -0.026        1.000         0.816  0.112
embed_length      -0.087        0.033        0.816         1.000 -0.468
tier               0.166       -0.086        0.112        -0.468  1.000

=== Key Correlations with avg_rating ===
  tier                   0.166  ↑
  num_ratings            0.058  ↑
  desc_length            0.006  ↑
  embed_length          -0.087  ↓

=== Tier Comparison ===
      avg_rating  num_ratings  desc_length
tier                                      
1           3.95    160390.91       818.04
2           4.06     87176.42       977.73


### Correlation Analysis - Key Insights

- **avg_rating vs num_ratings (0.058)**: almost no correlation - popularity does not equal quality, both signals needed independently
- **avg_rating vs tier (0.166)**: Tier 1 books (with CMU) have slightly lower avg rating (3.95 vs 4.06) - CMU tends to cover older classic literature with more polarized ratings
- **desc_length vs embed_length (0.816)**: strong correlation - description length is the dominant factor in embedding length as expected
- **embed_length vs tier (-0.468)**: Tier 2 books have longer embeddings - Goodreads descriptions for popular modern books tend to be more detailed
- **Tier 1 books have 84% more ratings on average** (160K vs 87K) - CMU covers more well-known, widely-read books
- **No single feature strongly predicts rating**: confirms that a hybrid approach (semantic + popularity + rating) is the right modeling strategy

---

In [6]:
# ═══════════════════════════════════════════════════════════════
# AUTHORS ANALYSIS
# ═══════════════════════════════════════════════════════════════

author_counts = df["authors"].value_counts()

print(f"Total unique authors : {len(author_counts):,}")
print(f"\nTop 15 most represented authors:")
for author, count in author_counts.head(15).items():
    bar = "█" * count
    print(f"  {author:<35} {count:>3}  {bar}")

print(f"\nAuthors with only 1 book : {(author_counts == 1).sum():,}")
print(f"Authors with 5+ books    : {(author_counts >= 5).sum():,}")
print(f"Authors with 10+ books   : {(author_counts >= 10).sum():,}")

Total unique authors : 4,955

Top 15 most represented authors:
  Stephen King                         56  ████████████████████████████████████████████████████████
  William Shakespeare                  38  ██████████████████████████████████████
  Terry Pratchett                      35  ███████████████████████████████████
  Agatha Christie                      33  █████████████████████████████████
  Anonymous                            28  ████████████████████████████
  Rick Riordan                         25  █████████████████████████
  John Grisham                         23  ███████████████████████
  Lucian Bane                          21  █████████████████████
  C.S. Lewis                           20  ████████████████████
  Roald Dahl                           18  ██████████████████
  Chuck Palahniuk                      18  ██████████████████
  Isaac Asimov                         18  ██████████████████
  Lee Child                            18  ██████████████████
  James Patter

### Authors Analysis - Key Insights

- **4,955 unique authors**: excellent diversity - no single author dominates the catalog
- **Stephen King leads with 56 books**: prolific authors well-represented across genres
- **"Anonymous" appears 28 times**: classical works without clear authorship - acceptable noise
- **3,671 authors with only 1 book (74%)**: long-tail distribution - typical for book catalogs
- **Only 66 authors with 10+ books**: a small core of prolific authors provides series-based recommendation opportunities
- **Implication for recommendations**: author signal is useful for diversity filtering but too sparse for collaborative filtering - content-based embedding remains the primary approach

---

## ✅ EDA Summary - Feature Engineering Decisions

| Finding | Decision for Next Step |
|---|---|
| Ratings well-distributed (mean 4.04) | Use Bayesian smoothing for weighted score |
| Popularity highly skewed | Log-transform num_ratings for normalization |
| "Audiobook", "Novels", "Historical" are noise genres | Remove in feature engineering |
| 611 unique genres, avg 6.7 per book | Extract top 50 genres as primary signal |
| Embedding text averages 1,251 chars | No truncation needed - within model limits |
| No strong single predictor of rating | Confirm hybrid approach (semantic + rating + popularity) |
| Tier 1 books are older, more popular classics | Weight tier in hybrid score |
| 4,955 authors, 74% with 1 book only | Author used for diversity, not as primary feature |

 **Next → `04_feature_engineering.ipynb`**